# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 05

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. Аналитик выполнил две строчки кода:
```python
s1 = df['balance']
s2 = df[['balance']]
```
В какой переменной (`s1` или `s2`) получен одномерный столбец (`pd.Series`), а в какой — двумерная таблица (`pd.DataFrame`) из одной колонки? В чем их главное отличие при обращении к методам?

**Ответ:**
* `s1 = df['balance']` возвращает одномерную структуру `pd.Series` (вектор значений).
* `s2 = df[['balance']]` (список в двойных квадратных скобках) возвращает двумерную таблицу `pd.DataFrame` из одной колонки.
К `Series` обращаются как к одномерному вектору, а к `DataFrame` применимы табличные методы (выборка нескольких колонок, отображение в виде таблицы).

---

#### 1.2. Аналитик вызвал `df.info()` для таблицы игровых сессий из 500 строк и увидел запись:
`client_rating: 425 non-null float64`.
Сколько сессий в таблице остались без клиентской оценки (содержат `NaN` / `None`), и какой метод Pandas позволяет подсчитать количество пропусков во всех столбцах?

**Ответ:**
Без оценки остались **75 сессий** (500 - 425 = 75). Для подсчета точного количества пропусков в каждом столбце используется метод `df.isna().sum()` (или `df.isnull().sum()`).

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["balance_thousands"] = (df["balance"] / 1000).astype(int)
```

**Ответ:**
Переводит баланс лицевого счета в целые тысячи рублей путем деления на 1000 и отбрасывания дробной части приведением к типу `int`.

---

#### 1.4. Аналитик выполнил расчет средней суммы оплат по играм:
```python
report = df.groupby('game_title')['total_paid'].mean()
```
Сколько строк получится в объекте `report` (столько же, сколько всех сессий, или столько, сколько уникальных игр)? Что стало индексом строк в полученном объекте?

**Ответ:**
В объекте `report` строк будет ровно столько, сколько **уникальных игр** в таблице (строки свернулись по уникальным значениям `game_title`). Новым индексом строк стали сами названия игр (`game_title`).

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `tournaments.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("tournaments.csv").parent if os.path.exists("tournaments.csv") else pathlib.Path("../variant_05")
df_audit = pd.read_csv(data_dir / "tournaments.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Суммарный призовой фонд: 52500.00")

Количество строк в таблице: 8
Столбцы датафрейма: ['tournament_id', 'tournament_name', 'game_title', 'prize_fund', 'entry_fee', 'start_date', 'min_rank']
Суммарный призовой фонд: 52500.00

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `bar_orders.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("bar_orders.csv").parent if os.path.exists("bar_orders.csv") else pathlib.Path(f"../variant_05")
df_data = pd.read_csv(data_dir / "bar_orders.csv", sep=";")

summary = df_data.groupby("category").agg({
    "total_amount": "sum",
    "quantity": "mean"
}).reset_index()

summary["quantity"] = summary["quantity"].round(2)
summary = summary.sort_values(by="total_amount", ascending=False).reset_index(drop=True)
display(summary)

,category,total_amount,quantity
0,Мерч,99400,1.48
1,Напитки,31620,1.50
2,Горячее,26680,1.42
3,Снеки,17720,1.41


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `members` поле `member_id` объявлено как `PRIMARY KEY` (первичный ключ).
Что произойдет, если система попытается добавить нового клиента с уже существующим номером `member_id = 1` или со значением `NULL`? Почему СУБД вернет ошибку?

**Ответ:**
СУБД заблокирует операцию и вернет ошибку нарушения ограничения целостности (`PRIMARY KEY / UNIQUE / NOT NULL constraint failed`). Значение первичного ключа обязано быть уникальным для каждой записи и не может быть пустым (`NULL`), так как оно однозначно идентифицирует строку.

---

#### 3.2. Чем случайное выполнение команды `DROP TABLE members;` опаснее для работы клуба, чем команда `DELETE FROM members;`?
Сможет ли программа регистрации сразу после этого вставить нового клиента?

**Ответ:**
`DELETE` удаляет только строки, сохраняя структуру таблицы, ее типы и ограничения (программа сможет сразу вставлять новых клиентов). `DROP TABLE` физически удаляет саму таблицу из схемы базы данных — приложение упадет с ошибкой «таблица не существует» (`no such table: members`).

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT min_rank, COUNT(*) AS tourn_cnt, SUM(prize_fund) AS total_prizes FROM tournaments GROUP BY min_rank ORDER BY total_prizes DESC;
```

**Ответ:**
Запрос группирует турниры по минимальному рангу участников (`min_rank`) и рассчитывает количество турниров и суммарный призовой фонд, упорядочивая по убыванию призовых.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_05")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT tournament_id, tournament_name, game_title, prize_fund, entry_fee, start_date
FROM tournaments
WHERE prize_fund >= 40000 AND entry_fee <= 500
ORDER BY prize_fund DESC, start_date ASC;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,tournament_id,tournament_name,game_title,prize_fund,entry_fee,start_date
0,101,Winter Cyber Cup: CS2,Counter-Strike 2,50000.0,500.0,2026-02-14
1,108,PUBG Survival Showdown,PUBG,45000.0,500.0,2026-05-29
2,104,Apex Legends Trio Battle,Apex Legends,40000.0,450.0,2026-04-05


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT discount_promo, COUNT(*) AS sessions_count, ROUND(SUM(total_paid), 2) AS total_revenue, ROUND(AVG(duration_hours), 2) AS avg_hours
FROM game_sessions
GROUP BY discount_promo
HAVING COUNT(*) > 50
ORDER BY sessions_count DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,discount_promo,sessions_count,total_revenue,avg_hours
0,0,395,186900.0,2.19
1,1,105,45562.0,2.28


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT p.pc_id, p.pc_number, p.zone_name, COUNT(s.session_id) AS sessions_count, ROUND(SUM(s.total_paid), 2) AS revenue
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
WHERE p.zone_name IN ('VIP', 'Bootcamp')
GROUP BY p.pc_id, p.pc_number, p.zone_name
ORDER BY revenue DESC
LIMIT 8;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,pc_id,pc_number,zone_name,sessions_count,revenue
0,24,PC-24,VIP,22,20387.0
1,19,PC-19,Bootcamp,23,14825.0
2,22,PC-22,Bootcamp,22,14498.0
3,23,PC-23,VIP,14,14322.0
4,17,PC-17,Bootcamp,28,13040.0
5,15,PC-15,Bootcamp,21,12623.0
6,26,PC-26,VIP,15,11248.0
7,16,PC-16,Bootcamp,17,10691.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT session_id, member_id, game_title, duration_hours, total_paid
FROM game_sessions
WHERE duration_hours > (SELECT AVG(duration_hours) FROM game_sessions)
ORDER BY duration_hours DESC
LIMIT 10;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,session_id,member_id,game_title,duration_hours,total_paid
0,20017,45,Apex Legends,6.0,900.0
1,20141,32,PUBG,6.0,702.0
2,20177,98,Valorant,6.0,2268.0
3,20230,26,Cyberpunk 2077,6.0,780.0
4,20253,68,Counter-Strike 2,6.0,2280.0
5,20291,98,PUBG,6.0,1500.0
6,20299,42,Warzone,6.0,1500.0
7,20322,1,Apex Legends,6.0,900.0
8,20003,13,Counter-Strike 2,5.0,1350.0
9,20025,91,GTA V,5.0,850.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Призовой фонд по игровым дисциплинам

**Найденные ошибки:**
1. Пропущена группировка: поле `game_title` выбирается вместе с агрегатными функциями `COUNT` и `SUM`, но предложение `GROUP BY game_title` отсутствует.

In [9]:
query_51_fixed = '''
SELECT 
    game_title,
    COUNT(*) AS tourn_count,
    SUM(prize_fund) AS total_prizes
FROM tournaments
GROUP BY game_title
ORDER BY total_prizes DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,game_title,tourn_count,total_prizes
0,Counter-Strike 2,2,170000.0
1,Dota 2,2,105000.0
2,PUBG,1,45000.0
3,Apex Legends,1,40000.0
4,Valorant,1,35000.0
5,Cyberpunk 2077,1,25000.0


#### 5.2. Призовой фонд турниров по дисциплинам и рангам

**Найденные ошибки:**
1. Агрегатная функция в `WHERE`: фильтрация по `SUM(prize_fund) > 20000` должна выполняться в `HAVING`.
2. Неполный `GROUP BY`: столбец `min_rank` выбран в `SELECT`, но отсутствует в списке группировки.

In [10]:
query_52_fixed = '''
SELECT 
    game_title,
    min_rank,
    SUM(prize_fund) AS total_prizes
FROM tournaments
GROUP BY game_title, min_rank
HAVING SUM(prize_fund) > 20000
ORDER BY total_prizes DESC;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,game_title,min_rank,total_prizes
0,Counter-Strike 2,Gold,120000.0
1,Dota 2,Silver,75000.0
2,Counter-Strike 2,Bronze,50000.0
3,PUBG,Silver,45000.0
4,Apex Legends,Bronze,40000.0
5,Valorant,Bronze,35000.0
6,Dota 2,Bronze,30000.0
7,Cyberpunk 2077,Bronze,25000.0


#### 5.3. Сравнение призовых фондов турниров со средним фондом

**Найденные ошибки:**
1. Неполный `GROUP BY`: поле `min_rank` указано в `SELECT`, но не включено в список группировки.
2. Порядок предложений и alias: секция `WHERE` написана после `GROUP BY` и использует псевдоним `avg_prize` вместо `HAVING AVG(prize_fund)`.
3. Многострочный подзапрос: наличие `GROUP BY game_title` в подзапросе возвращает несколько строк вместо одного скалярного значения.

In [11]:
query_53_fixed = '''
SELECT 
    game_title,
    min_rank,
    ROUND(AVG(prize_fund), 2) AS avg_prize
FROM tournaments
GROUP BY game_title, min_rank
HAVING AVG(prize_fund) > (
    SELECT AVG(prize_fund) 
    FROM tournaments
)
ORDER BY avg_prize DESC;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,game_title,min_rank,avg_prize
0,Counter-Strike 2,Gold,120000.0
1,Dota 2,Silver,75000.0
